# Clean REDCap address data, generate ADI, PM2.5 values
Read this SLAB for additional guidance: https://smithlab.slab.com/posts/gis-basics-s0755z2p 

Update the paths in the first code cell before running.

This notebook:
1. Reads a REDCap CSV (can be from RF1, Island screener, recruitment database, or whatever).
2. Reads a `.txt` subject list of interest.
3. Keeps only rows for those subjects.
4. Pulls address values from `participant_receipt_4` from the RF1 SRPAL report (can be changed depending on what you want address source column to be).
5. Does basic cleaning of address text into a geocoding-friendly format.
6. Writes the cleaned output to CSV.

You then will need to manually clean the address data. Once you have manually cleaned address data, go to "Cleaned data (for FIPS)" (to derive ADI) and/or "PM2.5 extraction" (using lat/lon data) and update the paths there. 

7. Extracts FIPS and latitude/longitude data for each participant to derive ADI and PM2.5 values. (Note: PM2.5 currently set to look for Years 2021-2023; can be amended to your use case)


In [ ]:
from pathlib import Path
import pandas as pd
import re

# ---- EDIT THESE PATHS ----
csv_path = Path("YourPath/RF1SocialRewardProce-AddressData_DATA_2026-05-03_2155.csv")
sublist_path = Path("/YourPath/sublist_address.txt")
out_path = Path("YourPath/cleaned_participant_receipt_addresses.csv")

# Column containing address values
address_col = "participant_receipt_4"

# Optional: keep only these REDCap events.
# Set to None if you want to search across all rows/events.
event_keep = None
# event_keep = ["subject_informatio_arm_1"]


## Read data

In [ ]:
df = pd.read_csv(csv_path, dtype=str).fillna("")

with open(sublist_path, "r") as f:
    sub_ids = [
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith("#")
    ]

print(f"Rows in CSV: {len(df):,}")
print(f"Subjects in txt file: {len(sub_ids):,}")
print(df.columns.tolist())

## Filter to subject list and pull address column

In [ ]:
df_sub = df[df["sub_id"].astype(str).isin(sub_ids)].copy()

if event_keep is not None:
    df_sub = df_sub[df_sub["redcap_event_name"].isin(event_keep)].copy()

address_df = (
    df_sub
    .loc[:, ["sub_id", "redcap_event_name", address_col]]
    .rename(columns={address_col: "address_raw"})
    .copy()
)

# Keep only rows with a non-empty address
address_df = address_df[address_df["address_raw"].str.strip().ne("")].copy()

print(f"Rows after subject filtering: {len(df_sub):,}")
print(f"Rows with non-empty addresses: {len(address_df):,}")
address_df.head()

## Clean address text

In [ ]:
def clean_address(x):
    """Basic cleanup for messy address strings."""
    if pd.isna(x):
        return ""

    x = str(x)

    # Normalize whitespace and line breaks
    x = x.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()

    # Remove common labels if they appear inside copied/pasted REDCap text
    x = re.sub(r"(?i)\b(address|street address|mailing address|receipt address)\s*[:\-]\s*", "", x)

    # Remove repeated punctuation / separators
    x = re.sub(r"\s*,\s*", ", ", x)
    x = re.sub(r",\s*,+", ", ", x)
    x = re.sub(r"\s{2,}", " ", x)

    # Standardize a few common abbreviations without getting too aggressive
    replacements = {
        r"\bApt\.?\b": "Apt",
        r"\bApartment\b": "Apt",
        r"\bSte\.?\b": "Suite",
        r"\bFl\.?\b": "Floor",
        r"\bP\.?O\.?\s*Box\b": "PO Box",
    }

    for pattern, repl in replacements.items():
        x = re.sub(pattern, repl, x, flags=re.IGNORECASE)

    return x.strip(" ,")


address_df["address_clean"] = address_df["address_raw"].apply(clean_address)

# Drop addresses that became empty after cleaning
address_df = address_df[address_df["address_clean"].str.strip().ne("")].copy()

address_df.head(20)

## Handle duplicates

This keeps one cleaned address per subject. If a subject has more than one unique address, those cases are flagged for review.


In [ ]:
# Count unique cleaned addresses per subject
address_counts = (
    address_df
    .groupby("sub_id")["address_clean"]
    .nunique()
    .reset_index(name="n_unique_addresses")
)

address_df = address_df.merge(address_counts, on="sub_id", how="left")

# Create audit dataframe
address_audit = address_df.sort_values(["sub_id", "redcap_event_name", "address_clean"])

# Keep all subjects from txt file, including missing addresses
all_subjects_df = pd.DataFrame({"sub_id": sub_ids})

address_one_per_subject = (
    address_audit
    .drop_duplicates(subset=["sub_id", "address_clean"])
    .sort_values(["sub_id", "redcap_event_name"])
    .drop_duplicates(subset=["sub_id"], keep="first")
    .loc[:, ["sub_id", "address_clean", "n_unique_addresses"]]
)

address_one_per_subject = (
    all_subjects_df
    .merge(address_one_per_subject, on="sub_id", how="left")
    .fillna({"address_clean": "", "n_unique_addresses": 0})
)

address_one_per_subject["needs_manual_address"] = address_one_per_subject["address_clean"].eq("")

# Subjects with more than one unique cleaned address
address_conflicts = (
    address_audit[address_audit["n_unique_addresses"] > 1]
    .loc[:, ["sub_id", "redcap_event_name", "address_raw", "address_clean"]]
    .drop_duplicates()
    .sort_values(["sub_id", "redcap_event_name"])
)

print(f"Subjects in final output: {address_one_per_subject['sub_id'].nunique():,}")
print(f"Subjects needing manual address: {address_one_per_subject['needs_manual_address'].sum():,}")
print(f"Subjects with multiple unique addresses: {address_conflicts['sub_id'].nunique():,}")

address_one_per_subject.head()

## Check missing subjects

In [ ]:
found_subjects = set(address_one_per_subject["sub_id"])
missing_subjects = sorted(set(sub_ids) - found_subjects)

missing_df = pd.DataFrame({"sub_id": missing_subjects})

print(f"Subjects from txt file with no non-empty address found: {len(missing_df):,}")
missing_df.head(50)

## Save outputs

In [ ]:
out_path.parent.mkdir(parents=True, exist_ok=True)

address_one_per_subject.to_csv(out_path, index=False)

audit_path = out_path.with_name(out_path.stem + "_audit.csv")
conflict_path = out_path.with_name(out_path.stem + "_conflicts.csv")
missing_path = out_path.with_name(out_path.stem + "_missing_subjects.csv")

address_audit.to_csv(audit_path, index=False)
address_conflicts.to_csv(conflict_path, index=False)
missing_df.to_csv(missing_path, index=False)

print("Wrote:")
print(out_path)
print(audit_path)
print(conflict_path)
print(missing_path)


## Cleaned data (for FIPS)

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path


# Paths (define to your use case)
old_path = Path("/Users/tur61139/sensitive/ADI_compiled_final.csv")
new_path = Path("/Users/tur61139/sensitive/cleaned_new_address.csv")
adi_dir = Path("/Users/tur61139/sensitive/NeighborhoodAtlas_ADI")
out_path = Path("/Users/tur61139/sensitive/ADI_compiled_final_new.csv")

# Load data
old = pd.read_csv(old_path, dtype=str).fillna("")
new = pd.read_csv(new_path, dtype=str).fillna("")

old["source_file"] = "old"
new["source_file"] = "new"

old["sub_id"] = old["sub_id"].astype(str).str.strip()
new["sub_id"] = new["sub_id"].astype(str).str.strip()

# Align columns
all_cols = list(dict.fromkeys(list(old.columns) + list(new.columns)))

for col in all_cols:
    if col not in old.columns:
        old[col] = ""
    if col not in new.columns:
        new[col] = ""

old = old[all_cols]
new = new[all_cols]

# Combine
df = pd.concat([old, new], ignore_index=True)

df["priority"] = df["source_file"].map({"old": 1, "new": 2})

df = (
    df
    .sort_values(["sub_id", "priority"])
    .drop_duplicates(subset=["sub_id"], keep="first")
    .drop(columns=["priority"])
    .reset_index(drop=True)
)

print("Total subjects after combine:", df["sub_id"].nunique())

# -------------------------
# Build geocoding address (if NO participant_receipt_4)
def make_address(row):
    # OLD ADI FILE; cross-referencing with this validated sheet
    if row["source_file"] == "old":
        parts = [
            row.get("address", "").strip(),
            row.get("Smarty_City", "").strip(),
            row.get("Smarty_State", "").strip(),
            row.get("Full_ZIP", "").strip()
        ]
        return ", ".join([p for p in parts if p])

    parts = [
        row.get("address", "").strip(),
        row.get("zip_code", "").strip()
    ]
    return ", ".join([p for p in parts if p])

df["geocode_address"] = df.apply(make_address, axis=1)

df["geocode_address"] = (
    df["geocode_address"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip(" ,")
)

# Census geocoder
def get_geo(addr):
    if not addr:
        return pd.Series({
            "lat": "",
            "lon": "",
            "block_group_fips": "",
            "match_status": "missing"
        })

    url = "https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress"

    params = {
        "address": addr,
        "benchmark": "Public_AR_Current",
        "vintage": "Current_Current",
        "format": "json",
        "layers": "all"
    }

    try:
        r = requests.get(url, params=params, timeout=20)
        matches = r.json()["result"]["addressMatches"]

        if not matches:
            return pd.Series({
                "lat": "",
                "lon": "",
                "block_group_fips": "",
                "match_status": "no_match"
            })

        m = matches[0]
        geos = m["geographies"]

        bg = geos.get("Census Block Groups", [{}])[0]

        return pd.Series({
            "lat": m["coordinates"]["y"],
            "lon": m["coordinates"]["x"],
            "block_group_fips": bg.get("GEOID", ""),
            "match_status": "matched"
        })

    except:
        return pd.Series({
            "lat": "",
            "lon": "",
            "block_group_fips": "",
            "match_status": "error"
        })

# Run geocoder
results = []

for i, addr in enumerate(df["geocode_address"]):
    results.append(get_geo(addr))

    if (i+1) % 10 == 0:
        print(f"{i+1}/{len(df)} done")

    time.sleep(0.2)

geo_df = pd.DataFrame(results)

df = pd.concat([df, geo_df], axis=1)

# Load ADI
adi_files = list(adi_dir.glob("*.csv"))

adi = pd.concat([pd.read_csv(f, dtype=str) for f in adi_files])

adi["FIPS"] = (
    adi["FIPS"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\.00$", "", regex=True)
)

adi["FIPS"] = adi["FIPS"].apply(
    lambda x: str(int(float(x))) if "E" in x.upper() else x
)

adi = (
    adi[["FIPS", "ADI_NATRANK", "ADI_STATERANK"]]
    .drop_duplicates(subset=["FIPS"])
)

# Merge ADI
df = df.drop(columns=["ADI_NATRANK", "ADI_STATERANK"], errors="ignore")

df_final = df.merge(
    adi,
    left_on="block_group_fips",
    right_on="FIPS",
    how="left"
).drop(columns=["FIPS"])

# Save
df_final.to_csv(out_path, index=False)

print("\nDONE")
print("Final subjects:", df_final["sub_id"].nunique())
print("Missing ADI:", df_final["ADI_NATRANK"].isna().sum())
print(df_final["match_status"].value_counts())

In [ ]:
import pandas as pd
from pathlib import Path

# paths
sublist_path = Path("YourPath/sublist_address.txt")
csv_path = Path("YourPath/ADI_compiled_final_new.csv")

# load sublist
with open(sublist_path) as f:
    sublist_ids = set(line.strip() for line in f if line.strip())

# load csv
df = pd.read_csv(csv_path, dtype=str)
df["sub_id"] = df["sub_id"].astype(str).str.strip()
csv_ids = set(df["sub_id"])

# compare
missing = sorted(sublist_ids - csv_ids)
extra = sorted(csv_ids - sublist_ids)

print(f"Subjects in sublist: {len(sublist_ids)}")
print(f"Subjects in CSV: {len(csv_ids)}")
print(f"Missing from CSV: {len(missing)}")
print(f"Extra in CSV: {len(extra)}")

# show missing subjects
if missing:
    print("\nMissing subject IDs:")
    print(missing[:50])

# save missing list
pd.DataFrame({"sub_id": missing}).to_csv(
    "/Users/tur61139/sensitive/missing_subjects.csv", index=False
)

## PM2.5 extraction

In [ ]:
import pandas as pd
import sys
# !{sys.executable} -m pip install xarray netCDF4
import xarray as xr
import numpy as np
import re
from pathlib import Path


# Paths
subject_path = Path("YourPath/ADI_compiled_final_new.csv")
pm25_dir = Path("YourPath/PM25-selected")
out_path = Path("YourPath/ADI_compiled_final_new_with_PM25.csv")

# Load subject data
df = pd.read_csv(subject_path, dtype=str)

df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Helper functions
def get_year_from_filename(path):
    name = path.name

    # catches 2021, 2022, 2023 if present
    m = re.search(r"20(21|22|23)", name)
    if m:
        return "20" + m.group(1)

    # catches filenames ending like 2112.nc, 2212.nc, 2312.nc
    m = re.search(r"(21|22|23)12", name)
    if m:
        return "20" + m.group(1)

    return None


def detect_lat_lon_names(ds):
    lat_candidates = ["lat", "latitude", "LAT", "Latitude"]
    lon_candidates = ["lon", "longitude", "LON", "Longitude"]

    lat_name = next((x for x in lat_candidates if x in ds.coords), None)
    lon_name = next((x for x in lon_candidates if x in ds.coords), None)

    if lat_name is None or lon_name is None:
        raise ValueError(f"Could not detect lat/lon names. Coordinates are: {list(ds.coords)}")

    return lat_name, lon_name


def detect_pm25_variable(ds):
    data_vars = list(ds.data_vars)

    # prefer likely PM2.5 variable names
    for v in data_vars:
        if "pm" in v.lower():
            return v

    # otherwise use first data variable
    if len(data_vars) == 1:
        return data_vars[0]

    raise ValueError(f"Could not detect PM2.5 variable. Data variables are: {data_vars}")


def extract_pm25_for_file(nc_file, df):
    ds = xr.open_dataset(nc_file)

    lat_name, lon_name = detect_lat_lon_names(ds)
    pm_var = detect_pm25_variable(ds)

    lons = ds[lon_name].values
    use_360_lon = np.nanmax(lons) > 180

    values = []

    for _, row in df.iterrows():
        lat = row["lat"]
        lon = row["lon"]

        if pd.isna(lat) or pd.isna(lon):
            values.append(np.nan)
            continue

        # convert negative longitude to 0-360 if raster uses that system
        lon_use = lon % 360 if use_360_lon else lon

        val = ds[pm_var].sel(
            {
                lat_name: lat,
                lon_name: lon_use
            },
            method="nearest"
        ).values

        values.append(float(np.squeeze(val)))

    ds.close()
    return values


# Find PM2.5 files
pm25_files = sorted(pm25_dir.glob("*.nc"))

print("PM2.5 files found:")
for f in pm25_files:
    print(f.name, "->", get_year_from_filename(f))

# Extract annual PM2.5 values
for f in pm25_files:
    year = get_year_from_filename(f)

    if year not in ["2021", "2022", "2023"]:
        print(f"Skipping file because year could not be detected: {f.name}")
        continue

    col = f"pm25_{year}"
    print(f"Extracting {col} from {f.name}")

    df[col] = extract_pm25_for_file(f, df)

# Average PM2.5 across 2021-2023 (can change to whatever months/years you desire/have data for, just make sure naming conventions align)
pm25_cols = [f"pm25_{year}" for year in ["2021", "2022", "2023"] if f"pm25_{year}" in df.columns]

df["pm25_2021_2023_mean"] = df[pm25_cols].mean(axis=1, skipna=True)

# Save
df.to_csv(out_path, index=False)

print("\nWrote:", out_path)
print("PM2.5 columns added:", pm25_cols + ["pm25_2021_2023_mean"])

df[["sub_id", "lat", "lon"] + pm25_cols + ["pm25_2021_2023_mean"]].head()